# Mastering Transformers: From Classification to Generation

Welcome to this comprehensive lab on the Transformer architecture. You'll go beyond basic classification and implement three distinct, powerful applications of Transformers, plus an exploration of how to improve performance:

1.  **Part 1: Multi-Class Classification**: You'll train a Transformer from scratch to classify Reuters newswires into 46 different topics.
2.  **Part 2: Sequence-to-Sequence Translation**: You'll build a full encoder-decoder Transformer to translate sentences from English to Spanish.
3.  **Part 3: Text Generation**: You'll use a large, pre-trained model (GPT-2) to generate creative text and control the output by changing key hyperparameters in the code.
4.  **Part 4: The Impact of Scale and Training**: You'll see firsthand how longer training times and more complex models lead to better results.

This notebook will provide a deep, hands-on understanding of how Transformers can be adapted to solve a wide range of NLP problems.

## Learning Objectives
- Implement an **encoder-only** Transformer for multi-class text classification.
- Build a full **encoder-decoder** Transformer from scratch for machine translation.
- Understand the roles of self-attention, cross-attention, and causal masking.
- Use a pre-trained generative model (GPT-2) for text creation.
- Tune generation hyperparameters like `temperature`, `top_k`, and `top_p` to control model creativity.
- Analyze how training duration and model complexity affect performance.

---

## Part 1: Multi-Class Classification

Our first task is to build an **encoder-only** Transformer. The model will read a news article and, by understanding the relationships between the words, classify it into one of 46 topics. We will use the built-in Keras Reuters dataset.

### 1.1 Setup and Data Preparation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from keras import layers
from keras.utils import pad_sequences
import re
import string

# Set random seeds for consistent results
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# -- Hyperparameters for Classification --
VOCAB_SIZE_CLS = 10000
MAX_LEN_CLS = 200
EMBED_DIM_CLS = 128
NUM_CLASSES = 46
BATCH_SIZE_CLS = 128

# Load and prepare the Reuters data
(x_train_cls, y_train_cls), (x_test_cls, y_test_cls) = keras.datasets.reuters.load_data(num_words=VOCAB_SIZE_CLS)
x_train_cls = pad_sequences(x_train_cls, maxlen=MAX_LEN_CLS)
x_test_cls = pad_sequences(x_test_cls, maxlen=MAX_LEN_CLS)

print(f"Training data shape: {x_train_cls.shape}")

### 1.2 Building the Transformer Encoder

In [ ]:
class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, maxlen, vocab_size, embed_dim):
        super().__init__()
        self.token_emb = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.pos_emb = layers.Embedding(input_dim=maxlen, output_dim=embed_dim)

    def call(self, x):
        maxlen = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=maxlen, delta=1)
        positions = self.pos_emb(positions)
        x = self.token_emb(x)
        return x + positions

class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super().__init__()
        self.att = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = keras.Sequential([
            layers.Dense(ff_dim, activation="relu"), 
            layers.Dense(embed_dim)
        ])
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(rate)
        self.dropout2 = layers.Dropout(rate)

    def call(self, inputs, training=False):
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

print("Custom Transformer layers defined. ✅")

### 1.3 Training the Classification Model

In [ ]:
NUM_HEADS_CLS = 2
FF_DIM_CLS = 32

inputs = layers.Input(shape=(MAX_LEN_CLS,))
embedding_layer = TokenAndPositionEmbedding(MAX_LEN_CLS, VOCAB_SIZE_CLS, EMBED_DIM_CLS)
x = embedding_layer(inputs)
transformer_block = TransformerBlock(EMBED_DIM_CLS, NUM_HEADS_CLS, FF_DIM_CLS)
x = transformer_block(x)
x = layers.GlobalAveragePooling1D()(x)
x = layers.Dropout(0.2)(x)
# The lines below have been corrected to properly call the layers on the input tensor 'x'
x = layers.Dense(64, activation="relu")(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

cls_model = keras.Model(inputs=inputs, outputs=outputs)
cls_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

print("Training classification model...")
history_cls = cls_model.fit(
    x_train_cls, y_train_cls,
    batch_size=BATCH_SIZE_CLS,
    epochs=20,
    validation_split=0.1,
    callbacks=[keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True)],
    verbose=1
)

loss, acc = cls_model.evaluate(x_test_cls, y_test_cls)
print(f"\nClassification Test Accuracy: {acc*100:.2f}%")

---

## Part 2: Sequence-to-Sequence Translation

Now we'll build a full **encoder-decoder** Transformer. The encoder will process an English sentence, and the decoder will use that information to generate the Spanish translation. This introduces **cross-attention**, where the decoder pays attention to the encoder's output, and **causal masking**, which prevents the decoder from cheating by looking ahead in the sequence it's trying to predict. 

### 2.1 Setup and Data Preparation

In [ ]:
import pathlib
import random

# Download and prepare the dataset
text_file = keras.utils.get_file(
    fname="spa-eng.zip",
    origin="http://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip",
    extract=True,
)
text_file = pathlib.Path(text_file).parent / "spa-eng" / "spa.txt"

with open(text_file) as f:
    lines = f.read().split("\n")[:-1]

text_pairs = []
for line in lines:
    eng, spa = line.split("\t")
    spa = "[start] " + spa + " [end]"
    text_pairs.append((eng, spa))

random.shuffle(text_pairs)
num_val_samples = int(0.15 * len(text_pairs))
num_train_samples = len(text_pairs) - 2 * num_val_samples
train_pairs = text_pairs[:num_train_samples]
val_pairs = text_pairs[num_train_samples : num_train_samples + num_val_samples]
test_pairs = text_pairs[num_train_samples + num_val_samples :]

print(f"{len(text_pairs)} total pairs")
print(f"{len(train_pairs)} training pairs")
print(f"{len(val_pairs)} validation pairs")
print(f"{len(test_pairs)} test pairs")

### 2.2 Vectorizing the Text Data

In [ ]:
# -- Hyperparameters for Translation --
VOCAB_SIZE_TRANS = 15000
MAX_LEN_TRANS = 60
EMBED_DIM_TRANS = 256
BATCH_SIZE_TRANS = 64

strip_chars = string.punctuation + "¿"
strip_chars = strip_chars.replace("[", "")
strip_chars = strip_chars.replace("]", "")

def custom_standardization(input_string):
    lowercase = tf.strings.lower(input_string)
    return tf.strings.regex_replace(lowercase, f"[{re.escape(strip_chars)}]", "")

# English (source) vectorization
eng_vectorization = layers.TextVectorization(
    max_tokens=VOCAB_SIZE_TRANS, output_mode="int", output_sequence_length=MAX_LEN_TRANS,
)
# Spanish (target) vectorization
spa_vectorization = layers.TextVectorization(
    max_tokens=VOCAB_SIZE_TRANS,
    output_mode="int",
    output_sequence_length=MAX_LEN_TRANS + 1,
    standardize=custom_standardization,
)

train_eng_texts = [pair[0] for pair in train_pairs]
train_spa_texts = [pair[1] for pair in train_pairs]
eng_vectorization.adapt(train_eng_texts)
spa_vectorization.adapt(train_spa_texts)

def format_dataset(eng, spa):
    eng = eng_vectorization(eng)
    spa = spa_vectorization(spa)
    return (
        {"encoder_inputs": eng, "decoder_inputs": spa[:, :-1]},
        spa[:, 1:],
    )

def make_dataset(pairs):
    eng_texts, spa_texts = zip(*pairs)
    eng_texts = list(eng_texts)
    spa_texts = list(spa_texts)
    dataset = tf.data.Dataset.from_tensor_slices((eng_texts, spa_texts))
    dataset = dataset.batch(BATCH_SIZE_TRANS)
    dataset = dataset.map(format_dataset, num_parallel_calls=tf.data.AUTOTUNE)
    return dataset.shuffle(2048).prefetch(16).cache()

train_ds = make_dataset(train_pairs)
val_ds = make_dataset(val_pairs)

print("Data vectorized. ✅")

### 2.3 Building the Encoder-Decoder Model

In [ ]:
class TransformerDecoder(layers.Layer):
    def __init__(self, embed_dim, latent_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.latent_dim = latent_dim
        self.num_heads = num_heads
        self.attention_1 = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim
        )
        self.attention_2 = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim
        )
        self.dense_proj = keras.Sequential(
            [layers.Dense(latent_dim, activation="relu"), layers.Dense(embed_dim),]
        )
        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()
        self.layernorm_3 = layers.LayerNormalization()
        self.supports_masking = True

    def call(self, inputs, encoder_outputs, mask=None):
        causal_mask = self.get_causal_attention_mask(inputs)
        if mask is not None:
            padding_mask = tf.cast(mask[:, tf.newaxis, :], dtype="int32")
            padding_mask = tf.minimum(padding_mask, causal_mask)

        # Causal Self-Attention
        attention_output_1 = self.attention_1(
            query=inputs, value=inputs, key=inputs, attention_mask=causal_mask
        )
        out_1 = self.layernorm_1(inputs + attention_output_1)

        # Cross-Attention
        attention_output_2 = self.attention_2(
            query=out_1,
            value=encoder_outputs,
            key=encoder_outputs,
            attention_mask=padding_mask,
        )
        out_2 = self.layernorm_2(out_1 + attention_output_2)

        proj_output = self.dense_proj(out_2)
        return self.layernorm_3(out_2 + proj_output)

    def get_causal_attention_mask(self, inputs):
        input_shape = tf.shape(inputs)
        batch_size, sequence_length = input_shape[0], input_shape[1]
        i = tf.range(sequence_length)[:, tf.newaxis]
        j = tf.range(sequence_length)
        mask = tf.cast(i >= j, dtype="int32")
        return mask

L_DIM = 2048 # Latent dimension for FFN
N_HEADS = 8  # Number of attention heads

# Encoder
encoder_inputs = keras.Input(shape=(None,), dtype="int64", name="encoder_inputs")
x = TokenAndPositionEmbedding(MAX_LEN_TRANS, VOCAB_SIZE_TRANS, EMBED_DIM_TRANS)(encoder_inputs)
encoder_outputs = TransformerBlock(EMBED_DIM_TRANS, N_HEADS, L_DIM)(x)
encoder = keras.Model(encoder_inputs, encoder_outputs)

# Decoder
decoder_inputs = keras.Input(shape=(None,), dtype="int64", name="decoder_inputs")
encoded_seq_inputs = keras.Input(shape=(None, EMBED_DIM_TRANS), name="decoder_state_inputs")
x = TokenAndPositionEmbedding(MAX_LEN_TRANS, VOCAB_SIZE_TRANS, EMBED_DIM_TRANS)(decoder_inputs)
x = TransformerDecoder(EMBED_DIM_TRANS, L_DIM, N_HEADS)(x, encoded_seq_inputs)
x = layers.Dropout(0.5)(x)
decoder_outputs = layers.Dense(VOCAB_SIZE_TRANS, activation="softmax")(x)
decoder = keras.Model([decoder_inputs, encoded_seq_inputs], decoder_outputs)

decoder_outputs = decoder([decoder_inputs, encoder_outputs])
transformer = keras.Model(
    [encoder_inputs, decoder_inputs], decoder_outputs, name="transformer"
)

transformer.summary()
transformer.compile(
    "rmsprop", loss="sparse_categorical_crossentropy", metrics=["accuracy"]
)

print("Translation model built. ✅")

### 2.4 Training and Inference

In [ ]:
print("Training translation model...")
transformer.fit(train_ds, epochs=30, validation_data=val_ds, 
              callbacks=[keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True)])

# Inference loop
spa_vocab = spa_vectorization.get_vocabulary()
spa_index_lookup = dict(zip(range(len(spa_vocab)), spa_vocab))
max_decoded_sentence_length = 20

def decode_sequence(input_sentence):
    tokenized_input_sentence = eng_vectorization([input_sentence])
    decoded_sentence = "[start]"
    for i in range(max_decoded_sentence_length):
        tokenized_target_sentence = spa_vectorization([decoded_sentence])[:, :-1]
        predictions = transformer([tokenized_input_sentence, tokenized_target_sentence])
        sampled_token_index = np.argmax(predictions[0, i, :])
        sampled_token = spa_index_lookup[sampled_token_index]
        if sampled_token == "[end]":
            break
        decoded_sentence += " " + sampled_token
    return decoded_sentence

# Test translation
test_eng_texts = [pair[0] for pair in test_pairs]
for _ in range(5):
    input_sentence = random.choice(test_eng_texts)
    print("-")
    print(f"English: {input_sentence}")
    print(f"Spanish: {decode_sequence(input_sentence)}")

---

## Part 3: Text Generation with Hyperparameter Tuning

Finally, we'll use a powerful, pre-trained model (GPT-2) for text generation. Training such models from scratch is computationally prohibitive, so we leverage the knowledge they have already gained from reading massive amounts of text. Our goal is to understand and control the generation process. We will edit parameters in the code to influence the model's creativity and coherence.

### 3.1 Loading the Pre-trained Model

In [ ]:
# We need to install the transformers library from Hugging Face
!pip install transformers -q

from transformers import TFGPT2LMHeadModel, GPT2Tokenizer

print("Loading GPT-2 model and tokenizer...")
# Load pre-trained model and tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = TFGPT2LMHeadModel.from_pretrained("gpt2", pad_token_id=tokenizer.eos_token_id)
print("Model loaded. ✅")

### 3.2 Generating Text with Different Parameters

In the cell below, you can change the variables to control the text generation. Here's what each parameter does:

- **`prompt_text`**: The starting text for the model.
- **`temperature`**: Controls randomness. Lower values (e.g., 0.7) make the model more deterministic and conservative, picking high-probability words. Higher values (e.g., 1.5) increase creativity and risk-taking.
- **`top_k`**: The model considers only the *k* most likely words at each step. A `top_k` of 50 means it will sample from the 50 most probable next words. Setting it to 0 disables this.
- **`top_p` (Nucleus Sampling)**: A more dynamic approach. The model samples from the smallest set of words whose cumulative probability exceeds `p`. A `top_p` of 0.9 means it will consider words until their combined probability is 90%.
- **`max_length`**: The total length of the generated text.

**Experiment by changing these values and re-running the cell!**

In [ ]:
# --- Generation Parameters (Change these!) ---
prompt_text = "In a shocking finding, scientist discovered a herd of unicorns living in a remote, previously unexplored valley, in the Andes Mountains."
max_length = 150
temperature = 1.0  # Try 0.7 for more focused text, or 1.5 for more wild text
top_k = 50
top_p = 0.95
# --- End of Parameters ---

print(f"--- Prompt ---\n{prompt_text}\n")

input_ids = tokenizer.encode(prompt_text, return_tensors='tf')

# Generate text using the specified parameters
sample_outputs = model.generate(
    input_ids,
    do_sample=True, # This must be True for sampling-based strategies
    max_length=max_length,
    temperature=temperature,
    top_k=top_k,
    top_p=top_p
)

output_text = tokenizer.decode(sample_outputs[0], skip_special_tokens=True)

print("--- Generated Text ---")
print(output_text)

---

## Part 4: The Impact of Scale and Training

A fundamental concept in deep learning is that performance scales with two key factors: **training time** and **model size**. In this section, we'll use our Reuters classification task to demonstrate these principles.

1.  **Longer Training**: We'll compare a model trained for only a few epochs versus one trained until convergence.
2.  **More Complex Architecture**: We'll build a deeper, more powerful Transformer and compare it against our original, simpler model.

### 4.1 Experiment 1: The Effect of Longer Training

In [ ]:
def build_simple_classifier():
    # Using the same simple architecture from Part 1
    NUM_HEADS_CLS = 2
    FF_DIM_CLS = 32
    
    inputs = layers.Input(shape=(MAX_LEN_CLS,))
    embedding_layer = TokenAndPositionEmbedding(MAX_LEN_CLS, VOCAB_SIZE_CLS, EMBED_DIM_CLS)
    x = embedding_layer(inputs)
    transformer_block = TransformerBlock(EMBED_DIM_CLS, NUM_HEADS_CLS, FF_DIM_CLS)
    x = transformer_block(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    
    model = keras.Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

# Model 1: Briefly Trained
print("Training model for a short duration (5 epochs)...")
model_brief = build_simple_classifier()
history_brief = model_brief.fit(x_train_cls, y_train_cls, batch_size=BATCH_SIZE_CLS, epochs=5, validation_split=0.1, verbose=1)
loss_brief, acc_brief = model_brief.evaluate(x_test_cls, y_test_cls, verbose=0)
print(f"Test Accuracy (Brief Training): {acc_brief*100:.2f}%\n")

# Model 2: Fully Trained
print("Training model for a long duration (up to 40 epochs)...")
model_full = build_simple_classifier()
history_full = model_full.fit(x_train_cls, y_train_cls, batch_size=BATCH_SIZE_CLS, epochs=40, validation_split=0.1, 
                            callbacks=[keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=4, restore_best_weights=True)], verbose=1)
loss_full, acc_full = model_full.evaluate(x_test_cls, y_test_cls, verbose=0)
print(f"Test Accuracy (Full Training): {acc_full*100:.2f}%\n")

# Plotting the results
plt.figure(figsize=(10, 5))
plt.plot(history_brief.history['val_accuracy'], label='Brief Training (5 Epochs)', linestyle='--')
plt.plot(history_full.history['val_accuracy'], label='Full Training (with Early Stopping)')
plt.title('Validation Accuracy: Longer Training Leads to Better Performance')
plt.xlabel('Epoch')
plt.ylabel('Validation Accuracy')
plt.legend()
plt.grid(True)
plt.show()

As you can see from the plot and the final scores, allowing the model to train for more epochs gives it the time it needs to converge on a much better solution. The briefly trained model was stopped before it could reach its full potential.

### 4.2 Experiment 2: The Effect of Architectural Complexity

In [ ]:
def build_complex_classifier():
    # A deeper and wider architecture
    NUM_HEADS_COMPLEX = 4  # More attention heads
    FF_DIM_COMPLEX = 64    # Wider feed-forward network
    NUM_BLOCKS = 3         # More Transformer blocks (deeper)
    
    inputs = layers.Input(shape=(MAX_LEN_CLS,))
    embedding_layer = TokenAndPositionEmbedding(MAX_LEN_CLS, VOCAB_SIZE_CLS, EMBED_DIM_CLS)
    x = embedding_layer(inputs)
    
    # Stack multiple Transformer blocks
    for _ in range(NUM_BLOCKS):
        x = TransformerBlock(EMBED_DIM_CLS, NUM_HEADS_COMPLEX, FF_DIM_COMPLEX)(x)
    
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    
    model = keras.Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

# Train the new, more complex model
print("Training complex model...")
model_complex = build_complex_classifier()
history_complex = model_complex.fit(x_train_cls, y_train_cls, batch_size=BATCH_SIZE_CLS, epochs=40, validation_split=0.1,
                                  callbacks=[keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=4, restore_best_weights=True)], verbose=1)
loss_complex, acc_complex = model_complex.evaluate(x_test_cls, y_test_cls, verbose=0)
print(f"Test Accuracy (Complex Model): {acc_complex*100:.2f}%")

### 4.3 Final Comparison

Let's summarize the results. A more complex model, given sufficient training time, has a higher **capacity** to learn the patterns in the data, leading to better performance. The trade-off is a significant increase in the number of trainable parameters, which requires more computational power and memory.

In [ ]:
summary_data = {
    'Model Configuration': [
        'Simple Model (Brief Training)',
        'Simple Model (Full Training)',
        'Complex Model (Full Training)'
    ],
    'Test Accuracy (%)': [acc_brief * 100, acc_full * 100, acc_complex * 100],
    '# Parameters': [
        model_brief.count_params(),
        model_full.count_params(),
        model_complex.count_params()
    ],
    'Training Epochs': [
        len(history_brief.epoch),
        len(history_full.epoch),
        len(history_complex.epoch)
    ]
}

summary_df = pd.DataFrame(summary_data).set_index('Model Configuration')
summary_df['Test Accuracy (%)'] = summary_df['Test Accuracy (%)'].map('{:.2f}'.format)
summary_df['# Parameters'] = summary_df['# Parameters'].map('{:,}'.format)

print("--- Performance Summary ---")
print(summary_df)

These experiments clearly show that both sufficient training time and a model architecture with adequate complexity are crucial for achieving high performance on a given task.

---

## Conclusion

In this lab, you have successfully implemented three different applications of the Transformer architecture and explored how to scale them.

1.  You built a **Transformer Encoder** to classify text, demonstrating the power of self-attention for understanding tasks.
2.  You constructed a full **Encoder-Decoder Transformer** for machine translation, learning how source and target sequences interact through cross-attention.
3.  You leveraged a massive, **pre-trained generative model** and learned to steer its output using key hyperparameters, gaining an intuition for controlling modern AI text generation.
4.  You demonstrated that **performance scales** with longer training and more complex models, revealing the fundamental trade-offs between accuracy and computational cost.

This journey from a simple classifier to an advanced generative tool showcases the remarkable flexibility and power of the Transformer architecture, which forms the foundation of modern NLP.